In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.vectorstores import FAISS


# ============================================================
# CONFIG (UPDATED FOR TESTING)
# ============================================================
HISTORY_CSV_PATH = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_QA.csv"
TEST_CSV_PATH = r"C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv"
INDEX_DIR = r"C:\Users\surya.adatravu\Documents\ContextRAG\vector_db"
SESSION_ID = "pdf_faiss_session"

# Optional: output results
RESULTS_OUT_CSV = r"C:\Users\surya.adatravu\Documents\ContextRAG\context_rag_test_results.csv"


# ============================================================
# LLM + EMBEDDINGS
# ============================================================
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


# ============================================================
# HISTORY STORE (IN-MEMORY)
# ============================================================
store = {}

def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


# ============================================================
# LOAD HISTORY FROM CSV (Q/A)
# ============================================================
def preload_history_from_csv(session_id: str, csv_path: str, max_rows=None, clear_existing=True):
    df = pd.read_csv(csv_path)
    if not {"Question", "Answer"}.issubset(df.columns):
        raise ValueError(f"History CSV must contain columns Question, Answer. Found: {list(df.columns)}")

    history = get_history(session_id)
    if clear_existing:
        history.clear()

    loaded = 0
    for _, row in df.iterrows():
        if max_rows is not None and loaded >= max_rows:
            break
        history.add_user_message(str(row["Question"]))
        history.add_ai_message(str(row["Answer"]))
        loaded += 1

    print(f"✅ Loaded {loaded} Q/A pairs into history (messages={len(history.messages)}) for session '{session_id}'")
    return history


# ============================================================
# LOAD TEST QUESTIONS CSV
# - Expect columns: Question, Answer
# - If your file uses different column names, update below.
# ============================================================
def load_test_questions(csv_path: str):
    df = pd.read_csv(csv_path)

    # Common variants
    possible_q = ["Question", "question", "Query", "query"]
    possible_a = ["Answer", "answer", "Expected", "expected_answer", "GroundTruth", "ground_truth"]

    q_col = next((c for c in possible_q if c in df.columns), None)
    a_col = next((c for c in possible_a if c in df.columns), None)

    if q_col is None:
        raise ValueError(f"Test CSV must include a question column (one of {possible_q}). Found: {list(df.columns)}")
    if a_col is None:
        raise ValueError(f"Test CSV must include an answer column (one of {possible_a}). Found: {list(df.columns)}")

    df = df.rename(columns={q_col: "Question", a_col: "ExpectedAnswer"})
    return df[["Question", "ExpectedAnswer"]]


# ============================================================
# LOAD FAISS
# ============================================================
def load_faiss(index_dir: str) -> FAISS:
    faiss_path = os.path.join(index_dir, "index.faiss")
    pkl_path = os.path.join(index_dir, "index.pkl")
    if not (os.path.exists(faiss_path) and os.path.exists(pkl_path)):
        raise FileNotFoundError(f"FAISS index not found in {index_dir}. Build it first.")
    return FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)


# ============================================================
# QUERY REWRITE (HISTORY-AWARE)
# ============================================================
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's latest question into a standalone search query.\n"
     "Use chat history only to resolve pronouns and references.\n"
     "Return ONLY the rewritten query."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

rewrite_chain = rewrite_prompt | llm
rewrite_with_history = RunnableWithMessageHistory(
    rewrite_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)


# ============================================================
# STRICT RAG ANSWER PROMPT (USES HISTORY + CONTEXT)
# ============================================================
rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict knowledge assistant.\n"
     "Use chat history only to interpret the question.\n"
     "You must ONLY answer using the provided CONTEXT.\n"
     "If answer is not in context, say exactly:\n"
     "\"I don't know from provided knowledge.\""),
    MessagesPlaceholder(variable_name="history"),
    ("human",
     "CONTEXT:\n{context}\n\n"
     "QUESTION:\n{question}\n\n"
     "Answer using ONLY the context.")
])

rag_chain = rag_prompt | llm
rag_with_history = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="history",
)


# ============================================================
# SCORING / VALIDATION HELPERS
# ============================================================
def similarity_score(text1: str, text2: str) -> float:
    v1 = embeddings.embed_query(text1)
    v2 = embeddings.embed_query(text2)
    return float(cosine_similarity([v1], [v2])[0][0])

def validate_pred_vs_expected(pred: str, expected: str, threshold: float = 0.82):
    score = similarity_score(pred, expected)
    return score >= threshold, score


# ============================================================
# ASK FUNCTION (RETURNS FULL RECORD FOR TESTING)
# ============================================================
def ask(question: str, vectorstore: FAISS, top_k=6, distance_threshold=0.75):
    rewritten = rewrite_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": SESSION_ID}}
    ).content.strip()

    docs_with_scores = vectorstore.similarity_search_with_score(rewritten, k=top_k)
    if not docs_with_scores:
        return {
            "question": question,
            "rewritten_query": rewritten,
            "answer": "I don't know from provided knowledge.",
            "retrieval_distance": None,
            "used_sources": "",
        }

    top_distance = float(docs_with_scores[0][1])
    if top_distance > distance_threshold:
        return {
            "question": question,
            "rewritten_query": rewritten,
            "answer": "I don't know from provided knowledge.",
            "retrieval_distance": top_distance,
            "used_sources": "",
        }

    retrieved_docs = [d for d, _ in docs_with_scores]

    # Track sources for analysis
    used_sources = sorted({
        f"{d.metadata.get('source_file', '?')}#page={d.metadata.get('page', '?')}"
        for d in retrieved_docs
    })

    context = "\n\n---\n\n".join(
        [
            f"[file={d.metadata.get('source_file','?')} page={d.metadata.get('page','?')}]\n{d.page_content}"
            for d in retrieved_docs
        ]
    )

    resp = rag_with_history.invoke(
        {"question": question, "context": context},
        config={"configurable": {"session_id": SESSION_ID}}
    )
    answer = resp.content.strip()

    return {
        "question": question,
        "rewritten_query": rewritten,
        "answer": answer,
        "retrieval_distance": top_distance,
        "used_sources": " | ".join(used_sources),
    }


# ============================================================
# RUN TEST SUITE
# ============================================================
def run_tests(
    vectorstore: FAISS,
    test_df: pd.DataFrame,
    validation_threshold: float = 0.82,
):
    rows = []
    pass_count = 0

    for i, r in test_df.iterrows():
        q = str(r["Question"])
        expected = str(r["ExpectedAnswer"])

        out = ask(q, vectorstore)

        pred = out["answer"]
        ok, sim = validate_pred_vs_expected(pred, expected, threshold=validation_threshold)

        row = {
            "idx": i,
            "question": q,
            "expected_answer": expected,
            "predicted_answer": pred,
            "cosine_similarity": sim,
            "pass": ok,
            "rewritten_query": out["rewritten_query"],
            "retrieval_distance": out["retrieval_distance"],
            "used_sources": out["used_sources"],
        }
        rows.append(row)

        pass_count += int(ok)

        # Optional: console progress
        print(f"[{i+1}/{len(test_df)}] pass={ok} sim={sim:.3f} dist={out['retrieval_distance']}")

    results_df = pd.DataFrame(rows)
    accuracy = pass_count / max(len(test_df), 1)

    print("\n====================")
    print(f"✅ Tests completed: {len(test_df)}")
    print(f"✅ Pass count     : {pass_count}")
    print(f"✅ Accuracy       : {accuracy:.3%}")
    print("====================\n")

    return results_df


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    # 1) Load history for contextual rewriting + interpretation
    preload_history_from_csv(SESSION_ID, HISTORY_CSV_PATH, max_rows=None, clear_existing=True)

    # 2) Load FAISS index
    vs = load_faiss(INDEX_DIR)

    # 3) Load tests
    test_df = load_test_questions(TEST_CSV_PATH)
    print(f"✅ Loaded {len(test_df)} test questions from: {TEST_CSV_PATH}")

    # 4) Run tests
    results = run_tests(vs, test_df, validation_threshold=0.82)

    # 5) Save results
    os.makedirs(os.path.dirname(RESULTS_OUT_CSV), exist_ok=True)
    results.to_csv(RESULTS_OUT_CSV, index=False)
    print(f"✅ Saved results to: {RESULTS_OUT_CSV}")


✅ Loaded 50 Q/A pairs into history (messages=100) for session 'pdf_faiss_session'
✅ Loaded 25 test questions from: C:\Users\surya.adatravu\Documents\ContextRAG\RA_FSM_RAG_Test_Questions.csv
[1/25] pass=False sim=0.272 dist=1.0582702159881592
[2/25] pass=False sim=0.184 dist=0.9055246114730835
[3/25] pass=False sim=0.150 dist=0.9286885261535645
[4/25] pass=False sim=0.217 dist=0.951801061630249
[5/25] pass=False sim=0.100 dist=1.2967432737350464
[6/25] pass=False sim=0.161 dist=1.3182315826416016
[7/25] pass=False sim=0.177 dist=0.861478865146637
[8/25] pass=False sim=0.155 dist=1.095931053161621
[9/25] pass=False sim=0.231 dist=0.8681557774543762
[10/25] pass=False sim=0.202 dist=1.2409489154815674
[11/25] pass=False sim=0.152 dist=0.9988982081413269
[12/25] pass=False sim=0.165 dist=0.9619455337524414
[13/25] pass=False sim=0.161 dist=0.8536734580993652
[14/25] pass=False sim=0.110 dist=1.102959156036377
[15/25] pass=False sim=0.178 dist=1.4031575918197632
[16/25] pass=False sim=0.144